In [1]:
!pip install transformers torch

In [2]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.nn.functional as F

In [3]:
class RandomLM(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embed = nn.Embedding(vocab_size,16)
    self.linear = nn.Linear(16, vocab_size)

  def forward(self,x):
    x = self.embed(x)
    x = x.mean(dim = 0)
    return self.linear(x)

sentences = [
    "i love ai",
    "i hate ai",
    "pizza is great"
]

vocab = ["i","love","hate","ai","pizza","is","great"]
word2idx = {w : i for i,w in enumerate(vocab)}
idx2word = {i: w for w,i in word2idx.items()}

print(word2idx)
print(idx2word)

{'i': 0, 'love': 1, 'hate': 2, 'ai': 3, 'pizza': 4, 'is': 5, 'great': 6}
{0: 'i', 1: 'love', 2: 'hate', 3: 'ai', 4: 'pizza', 5: 'is', 6: 'great'}


In [4]:
model_random = RandomLM(len(vocab))

def predict_random(sentence):
  tokens = torch.tensor([word2idx[w] for w in sentence.split()])
  with torch.no_grad():
    out = model_random(tokens)
  return idx2word[out.argmax().item()]

print("i love ->",predict_random("i love"))
print("ai is ->",predict_random("ai is"))

i love -> ai
ai is -> ai


In [5]:
data = [
    ("i love","ai"),
    ("ai is","great"),
    ("i hate", "pizza")
]

pairs = [(torch.tensor([word2idx[w] for w in inp.split()]),torch.tensor(word2idx[out])) for inp, out in data]
print(pairs)

[(tensor([0, 1]), tensor(3)), (tensor([3, 5]), tensor(6)), (tensor([0, 2]), tensor(4))]


In [6]:
optimizer = torch.optim.Adam(model_random.parameters(),lr = 0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(200):
  total_loss = 0
  for inp, target in pairs :
    out = model_random(inp)
    loss = loss_fn(out.unsqueeze(0),target.unsqueeze(0))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss = total_loss + loss.item()

  if epoch % 10 == 0 :
    print(f"eopch {epoch}, loss = {total_loss:.4f}")

eopch 0, loss = 4.8026
eopch 10, loss = 1.1115
eopch 20, loss = 0.3175
eopch 30, loss = 0.1316
eopch 40, loss = 0.0728
eopch 50, loss = 0.0470
eopch 60, loss = 0.0331
eopch 70, loss = 0.0247
eopch 80, loss = 0.0192
eopch 90, loss = 0.0154
eopch 100, loss = 0.0126
eopch 110, loss = 0.0106
eopch 120, loss = 0.0090
eopch 130, loss = 0.0078
eopch 140, loss = 0.0068
eopch 150, loss = 0.0059
eopch 160, loss = 0.0053
eopch 170, loss = 0.0047
eopch 180, loss = 0.0042
eopch 190, loss = 0.0038


In [7]:
print("i love ->",predict_random("i love"))
print("ai is ->", predict_random("ai is"))

i love -> ai
ai is -> great


In [8]:
model_name = "google/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [9]:
prompt = "AI is"

inputs = tokenizer(prompt, return_tensors = "pt")
outputs = model.generate(**inputs,max_length = 50)

print(tokenizer.decode(outputs[0]))

<bos>AI is no longer just a futuristic concept. It's already transforming various aspects of our lives. 

**Here's a breakdown of AI's current impact:**

*   **Automation:** Tasks that were once done manually are now


**Tokenization, Encoding & Decoding**

In [10]:
text = "AI will change everything"
tokens = tokenizer(text)
print(tokens)

{'input_ids': [2, 12553, 795, 2352, 4326], 'attention_mask': [1, 1, 1, 1, 1]}


In [11]:
decoded = tokenizer.decode(tokens["input_ids"])
print(decoded)

<bos>AI will change everything


In [12]:
decoded = tokenizer.decode(481)
print(decoded)

�


In [13]:
for token_id in tokens["input_ids"]:
  print(token_id,"->",tokenizer.decode([token_id]))

2 -> <bos>
12553 -> AI
795 ->  will
2352 ->  change
4326 ->  everything


**How Model works, Logits and Probabilities**

In [14]:
inputs = tokenizer("AI is",return_tensors = "pt")
with torch.no_grad():
  outputs = model(**inputs)
logits = outputs.logits
print("Logits Shape : ",logits.shape)
print("logits",logits)


Logits Shape :  torch.Size([1, 3, 262144])
logits tensor([[[-14.1875,  -1.1172,   4.6562,  ..., -14.6250, -14.6875, -14.6250],
         [-16.2500,  -0.1514,   3.1250,  ..., -16.8750, -16.7500, -16.7500],
         [-17.1250,  -3.4219,   0.3711,  ..., -18.3750, -18.2500, -18.1250]]],
       dtype=torch.bfloat16)


**Convert logits -> Probabilities**

In [15]:
last_token_logits = logits[0,-1]
probs = F.softmax(last_token_logits, dim = 0)

top_k = torch.topk(probs, k = 5)

for i in range(5):
  token_id = top_k.indices[i].item()
  print(tokenizer.decode([token_id]),"->",top_k.values[i].item())

 rapidly -> 0.625
 transforming -> 0.12353515625
 increasingly -> 0.03125
 changing -> 0.03125
 poised -> 0.0189208984375


**Control and Governance**

In [16]:
outputs = model.generate(
    **inputs,
    max_length = 30,
    temperature = 0.1
)

print("Low temp ->",tokenizer.decode(outputs[0]))

Low temp -> <bos>AI is rapidly changing the world, and its impact on education is particularly profound. From personalized learning to automated grading, AI tools are becoming increasingly integrated


In [17]:
outputs = model.generate(
    **inputs,
    max_length = 30,
    temperature = 1.5
)

print("Low temp ->",tokenizer.decode(outputs[0]))

Low temp -> <bos>AI is transforming nearly every aspect of our lives, and **natural language processing (NLP)** is at its core. From chatbots and virtual assistants to


In [18]:
outputs = model.generate(
    **inputs,
    max_length = 30,
    top_k = 10
)

print("Low temp ->",tokenizer.decode(outputs[0]))

Low temp -> <bos>AI is rapidly transforming industries, but it’s important to understand the potential risks alongside the benefits.  One significant area of concern is the rise


In [19]:
prompt = "Quantum pizza theory suggests"

inputs = tokenizer(prompt,return_tensors = "pt")
outputs = model.generate(**inputs, max_length = 40)

print(tokenizer.decode(outputs[0]))

<bos>Quantum pizza theory suggests that the pizza flavor is determined by the position of the cheese and pepperoni.

This is based on a complex observation of the quantum foam in a pizza box.

The cheese


In [20]:
prompt = "Article 570 section c of 3rd clause states that"
inputs = tokenizer(prompt,return_tensors = "pt")
outputs = model.generate(**inputs, max_length = 40)

print(tokenizer.decode(outputs[0]))

<bos>Article 570 section c of 3rd clause states that “The name of the company is the name of the applicant.”

This is unusual and potentially problematic, as it implies a
